# Evaluation Results Viewer
Reads one or more `eval_results.json` files (produced by `tools/test_tracked.py`) and presents the data as pandas DataFrames.

In [ ]:
import json
import os

import matplotlib.pyplot as plt
import pandas as pd

# ── Configuration ────────────────────────────────────────────────────────────
# One path or a list of paths. Each file has the same nested JSON structure.
RESULTS_FILES = [
    "/local/home/nkoefarago/mmpose/benchmark/results/20260528_coco_e2e.json",
    "/local/home/nkoefarago/mmpose/benchmark/results/20260527_coco_topdown.json",
    # "/local/home/nkoefarago/mmpose/benchmark/results/20260601_ochuman_e2e.json",
    # "/local/home/nkoefarago/mmpose/benchmark/results/20260601_ochuman_topdown.json",
]
# Examples:
# RESULTS_FILES = '/absolute/path/to/eval_results.json'
# RESULTS_FILES = [
#     '/path/to/eval_results.json',
#     '/path/to/other_eval_results.json',
# ]

if isinstance(RESULTS_FILES, str):
    RESULTS_FILES = [RESULTS_FILES]


def load_eval_results(path):
    with open(path) as f:
        raw = json.load(f)
    records = []
    for model_name, variants in raw.items():
        for variant, runs in variants.items():
            for run in runs:
                record = {
                    'model': model_name,
                    'variant': variant,
                    'timestamp': pd.Timestamp(run['timestamp']),
                    'config': run.get('config', ''),
                    'checkpoint': run.get('checkpoint', ''),
                    'source_file': path,
                }
                record.update(run.get('metrics', {}))
                records.append(record)
    return records


# ── Load & flatten ───────────────────────────────────────────────────────────
records = []
for path in RESULTS_FILES:
    if not os.path.isfile(path):
        raise FileNotFoundError(f'Results file not found: {path}')
    records.extend(load_eval_results(path))

df = pd.DataFrame(records)

# Derive all metric columns (everything after the fixed columns)
ID_COLS = ["model", "variant", "timestamp"]
META_COLS = ['config', 'checkpoint', 'source_file']
METRIC_COLS = [c for c in df.columns if c not in (ID_COLS + META_COLS)]

# Fixed plot color per known model family (matplotlib tab20); extend order when adding families.
_MODEL_COLOR_ORDER = [
    'DARK', 'HRFormer', 'HRNet', 'MSPN', 'PCT', 'PETR', 'RSN',
    'RTMPose', 'Sapiens', 'SimCC', 'UDP', 'ViTPose', 'YOLO-Pose', 'YOLO26-Pose',
]
_model_cmap = plt.get_cmap('tab20')
MODEL_COLORS = {m: _model_cmap(i) for i, m in enumerate(_MODEL_COLOR_ORDER)}

print(
    f'Loaded {len(df)} evaluation run(s) from {len(RESULTS_FILES)} file(s) '
    f'| metrics: {METRIC_COLS}'
)

## All evaluation entries

In [ ]:
display_cols = ID_COLS + METRIC_COLS

all_entries = (
    df[display_cols]
    .sort_values(['model', 'variant', 'timestamp'])
    .reset_index(drop=True)
)

display(
    all_entries.style
    .format({c: '{:.4f}' for c in METRIC_COLS}, na_rep='—')
    .set_caption('All evaluation runs')
    .set_table_styles([{'selector': 'caption',
                        'props': [('font-size', '14px'), ('font-weight', 'bold')]}])
)

## Latest result per model / variant

In [ ]:
DISPLAY_COLS = ["model", "variant"] + METRIC_COLS

AP_AR_COLS = [c for c in METRIC_COLS if '/AP' in c or '/AR' in c]
OTHER_METRIC_COLS = [c for c in METRIC_COLS if c not in AP_AR_COLS]

fmt = {c: (lambda x: f'{x*100:.1f}' if pd.notna(x) else '—') for c in AP_AR_COLS}
fmt.update({c: '{:.4f}' for c in OTHER_METRIC_COLS})

latest = (
    df.sort_values('timestamp')
    .groupby(['model', 'variant'], sort=False)
    .last()
    .reset_index()
)[DISPLAY_COLS].sort_values(['model', 'variant']).reset_index(drop=True)

display(
    latest.style
    .format(fmt, na_rep='—')
    .set_caption('Latest result per model / variant')
    .set_table_styles([{'selector': 'caption',
                        'props': [('font-size', '14px'), ('font-weight', 'bold')]}])
)

## Best `coco/AP` run per model

In [ ]:
AP_COL = 'coco/AP'

if AP_COL not in df.columns:
    print(f"Column '{AP_COL}' not found in results. "
          "Available metric columns:", METRIC_COLS)
else:
    best_ap = (
        df.dropna(subset=[AP_COL])
        .sort_values(AP_COL, ascending=False)
        .groupby('model', sort=False)
        .first()
        .reset_index()
    )[ID_COLS + METRIC_COLS].sort_values('model').reset_index(drop=True)

    display(
        best_ap.style
        .format({c: '{:.4f}' for c in METRIC_COLS}, na_rep='—')
        .set_caption(f'Best {AP_COL} run per model (variant + timestamp shown)')
        .set_table_styles([{'selector': 'caption',
                            'props': [('font-size', '14px'), ('font-weight', 'bold')]}])
    )

## AP vs e2e FPS

In [ ]:
import hashlib

import matplotlib.pyplot as plt

AP_COL = 'coco/AP'
FPS_COL = 'perf/e2e/fps'
# Set to a positive int to label only the N highest-AP points (0 = no point labels).
ANNOTATE_TOP_N = 0

def color_for_model(model: str):
    """Fixed color for known families; stable hash fallback for others."""
    if model in MODEL_COLORS:
        return MODEL_COLORS[model]
    digest = hashlib.md5(model.encode()).hexdigest()
    n_reserved = len(_MODEL_COLOR_ORDER)
    idx = n_reserved + (int(digest, 16) % (_model_cmap.N - n_reserved))
    return _model_cmap(idx)


missing = [c for c in (AP_COL, FPS_COL) if c not in df.columns]
if missing:
    print(f'Missing columns for plot: {missing}')
    print('Available metric columns:', METRIC_COLS)
else:
    plot_df = (
        df.sort_values('timestamp')
        .groupby(['model', 'variant'], sort=False)
        .last()
        .reset_index()
        .dropna(subset=[AP_COL, FPS_COL])
    )

    if plot_df.empty:
        print(f'No rows with both {AP_COL} and {FPS_COL}.')
    else:
        models = plot_df['model'].unique()

        fig, ax = plt.subplots(figsize=(15, 9))
        for model in models:
            if model == 'Sapiens':
                continue
            sub = plot_df[plot_df['model'] == model]
            ax.scatter(
                sub[FPS_COL],
                sub[AP_COL],
                s=60,
                alpha=0.85,
                color=color_for_model(model),
                label=model,
            )

        if ANNOTATE_TOP_N > 0:
            top = plot_df.nlargest(ANNOTATE_TOP_N, AP_COL)
            for _, row in top.iterrows():
                ax.annotate(
                    f"{row['variant']}",
                    (row[FPS_COL], row[AP_COL]),
                    xytext=(4, 4),
                    textcoords='offset points',
                    fontsize=7,
                    color=color_for_model(row['model']),
                )

        ax.set_xlabel('e2e FPS')
        ax.set_ylabel('coco/AP')
        ax.set_title('Accuracy vs end-to-end throughput (latest run per model / variant)')
        ax.grid(True, alpha=0.3)
        ax.legend(
            title='model',
            bbox_to_anchor=(1.02, 1),
            loc='upper left',
            fontsize=8,
            framealpha=0.9,
        )
        fig.tight_layout()

        # Variant names for each point (no overlap on the chart).
        display(
            plot_df[['model', 'variant', FPS_COL, AP_COL]]
            .sort_values([AP_COL, FPS_COL], ascending=[False, False])
            .reset_index(drop=True)
            .style.format({AP_COL: '{:.3f}', FPS_COL: '{:.1f}'})
            .set_caption('Points on plot (hover-free lookup)')
       
        )
        plt.show()